In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [16]:
pedidos = ['um','dois','tres']
caracteristicas = ['largura','quantidade']

largura_padrao = 4200

informacoes = {
    pedidos[0]:{
        'largura':1.3*1000,
        'quantidade':100
    },
    pedidos[1]:{
        'largura':1.4*1000,
        'quantidade':150
    },
    pedidos[2]:{
        'largura':1.6*1000,
        'quantidade':120
    },

}

In [51]:
model = pyo.ConcreteModel()

model.pedidos = pyo.Set(initialize=pedidos)
model.caracteristicas = pyo.Set(initialize=caracteristicas)

model.y = pyo.Var(bounds=(0,None), domain=pyo.NonNegativeReals)

# def obj(model):
#     return sum(model.x[p] for p in model.pedidos)
# model.ojetivo = pyo.Objective(rule=obj, sense=pyo.maximize)

# def restricao(model):
#     return sum(model.x[p]* informacoes[p]['largura'] for p in model.pedidos) <= largura_padrao
# model.restricao = pyo.Constraint(rule=restricao)

def obj(model):
    return model.y
model.objetivo = pyo.Objective(rule=obj, sense=pyo.minimize)

def restricao(model):
    return sum(informacoes[p]['largura']*informacoes[p]['quantidade'] for p in model.pedidos) <= 4200*model.y
model.restricao = pyo.Constraint(rule=restricao)


In [52]:
model.pprint()

2 Set Declarations
    caracteristicas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'largura', 'quantidade'}
    pedidos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'um', 'dois', 'tres'}

1 Var Declarations
    y : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :  None :  None : False :  True : NonNegativeReals

1 Objective Declarations
    objetivo : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : minimize :          y

1 Constraint Declarations
    restricao : Size=1, Index=None, Active=True
        Key  : Lower    : Body   : Upper : Active
        None : 532000.0 : 4200*y :  +Inf :   True

5 Declarations: pedidos caracteristicas y objetivo restricao


In [53]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmpa79j0hy6.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpqw202il7.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpqw202il7.pyomo.lp
Objective sense      : Minimize
Variables            :       1
Objective nonzeros   :       1
Linear constraints   :       1  [Greater: 1]
  Nonzeros           :       1
  RHS nonzeros       :       1

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 1.000000         Max   : 1.000000 

In [55]:
# for m in model.x:
#     print(f"{m}: {pyo.value(model.x[m])}")

print(f"{pyo.value(model.y)}")

126.66666666666667


#### Nao acho que seja isso que esteja pedindo no problema

##### acho que é saber quais possiveis combinações lado a lado pode ter 
#### preciso identificar possiveis combinações e testar com elas

In [ ]:
# pedido 1 = 1300
# pedido 2 = 1400
# pedido 3 = 1600


possibilidades={
    'p1_quantidade_refugo':{
        pedidos[0]:3,
        pedidos[1]:0,
        pedidos[2]:0,
        'refugo':300,
    },
    'p2_quantidade_refugo':{
        pedidos[0]:2,
        pedidos[1]:1,
        pedidos[2]:0,
        'refugo':200,
    },

}

# Acho que seria isso, mas da mto trabalho fazer na mao
# tem q ter um jeito mais prático, mas n to vendo agora


In [ ]:
model2 = pyo.ConcreteModel()


model2.pedidos = pyo.Set(initialize=pedidos)
model2.caracteristicas = pyo.Set(initialize=caracteristicas)
model2.possibilidades = pyo.RangeSet(0,7)

model2.x = pyo.Var(model2.possibilidades, bounds=(0,None), domain=pyo.NonNegativeReals)
# model2.y = pyo.Var(bounds=(0,None), domain=pyo.NonNegativeReals)
lista_refugos_para_cada_possibilidade = [
    100,200,300,400,500,600,700,800
]

def objetivo(model2):
    return sum(model2.x[p]*lista_refugos_para_cada_possibilidade[p] for p in model2.possibilidades)
model2.restricao = pyo.Objective(rule=objetivo, sense=pyo.minimize)

def rest(model2):
    return sum()


In [76]:
model2.pprint()

2 Set Declarations
    caracteristicas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'largura', 'quantidade'}
    pedidos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'um', 'dois', 'tres'}

1 RangeSet Declarations
    possibilidades : Dimen=1, Size=8, Bounds=(0, 7)
        Key  : Finite : Members
        None :   True :   [0:7]

1 Var Declarations
    x : Size=8, Index=possibilidades
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          0 :     0 :  None :  None : False :  True : NonNegativeReals
          1 :     0 :  None :  None : False :  True : NonNegativeReals
          2 :     0 :  None :  None : False :  True : NonNegativeReals
          3 :     0 :  None :  None : False :  True : NonNegativeReals
          4 :     0 :  None :  None : False :  True : NonNegativeReals
          5 :     0 :  None :  Non